# Notebook 03a: Binder Contact Filter

Filters binders by whether they contact defined Tau K18 epitope residues.
Run **once for Forge**, once for **BindCraft** by changing `METHOD` in the config cell.

```
[NB02b] bindcraft_CONFORMER.zip  ──┐
[NB02c] forge_designs.zip        ──┤  upload here (one method at a time)
                                    ↓
[THIS NOTEBOOK]
  • Parses binder+target complex PDBs
  • Cα–Cα contact analysis against PHF6* / PHF6 / jR2R3
  • Merges contact scores into input CSV (preserves sequence, iptm, plddt)
    ↓
{method}_binder_filter_passing.csv  ← NB03 input
```

**Note on Forge PDBs**: Forge outputs binder-only structures (no target chain).  
Set `HAS_COMPLEX_PDB = False` for Forge — the filter will use pLDDT only,  
and contact data will be computed in NB03 once target conformers are available.

## 0. Install

In [ ]:
%%capture
!pip install biopython pandas matplotlib

## 1. Configuration

Only edit this cell.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────

METHOD = 'forge'        # 'forge' or 'bindcraft'

# Does the zip contain complex PDBs (binder chain B + target chain A)?
#   BindCraft → True  (outputs full complex)
#   Forge     → False (ESMFold outputs binder-only; contact filter skipped)
HAS_COMPLEX_PDB = False

# Zip file names — from NB02b/02c downloads
# For BindCraft: one zip per conformer, upload the one you want to filter
# For Forge: one zip covering all conformers
PDB_ZIP = 'forge_designs.zip'         # change to e.g. 'bindcraft_rank01.zip'
CSV_INPUT = 'forge_binder_filter_passing.csv'  # CSV from NB02b/02c (has name, sequence, iptm)

TARGET_CHAIN = 'A'     # target chain ID in complex PDBs
BINDER_CHAIN = 'B'     # binder chain ID in complex PDBs

# Epitope ranges on target chain (K18 1-indexed residue numbers)
EPITOPE_RANGES = [
    (2,  7,  'PHF6* VQIINK'),
    (33, 38, 'PHF6 VQIVYK'),
    (52, 70, 'jR2R3'),
]

# Filter thresholds
MIN_EPITOPE_CONTACTS   = 1      # binder must touch ≥ this many epitope residues
CONTACT_CUTOFF_A       = 8.0    # Cα–Cα distance threshold (Å)
MIN_INTERFACE_PLDDT    = 70.0   # mean pLDDT of contacting binder residues
MIN_PLDDT_FORGE        = 70.0   # pLDDT filter for Forge (no complex PDB available)

# ─────────────────────────────────────────────────────────────────────────────
import os
os.makedirs('binder_pdbs', exist_ok=True)
os.makedirs('binders_passing', exist_ok=True)

epitope_res_map = {}
for start, end, label in EPITOPE_RANGES:
    for r in range(start, end + 1):
        epitope_res_map[r] = label

print(f'Method         : {METHOD}')
print(f'Complex PDBs   : {HAS_COMPLEX_PDB}')
print(f'Contact cutoff : {CONTACT_CUTOFF_A} Å')
print(f'Min epi contacts: {MIN_EPITOPE_CONTACTS}')
print(f'Min iface pLDDT: {MIN_INTERFACE_PLDDT}')
print(f'Epitope ranges :')
for start, end, label in EPITOPE_RANGES:
    print(f'  {label:20s} residues {start}–{end}')

## 2. Upload files

In [ ]:
from google.colab import files as _colab_files
import zipfile, glob, pandas as pd

print(f'Upload {PDB_ZIP} and {CSV_INPUT} from NB02b/02c...')
uploaded = _colab_files.upload()

# Extract PDB zip
if PDB_ZIP in uploaded or os.path.exists(PDB_ZIP):
    with zipfile.ZipFile(PDB_ZIP) as z:
        z.extractall('binder_pdbs')
    print(f'Extracted {PDB_ZIP} → binder_pdbs/')
else:
    print(f'WARNING: {PDB_ZIP} not found — check filename')

pdb_files = sorted(
    glob.glob('binder_pdbs/**/*.pdb', recursive=True) +
    glob.glob('binder_pdbs/*.pdb')
)
print(f'PDB files found: {len(pdb_files)}')

# Load input CSV (has name, sequence, iptm, plddt from NB02b/02c)
if CSV_INPUT in uploaded or os.path.exists(CSV_INPUT):
    input_df = pd.read_csv(CSV_INPUT)
    print(f'Input CSV: {len(input_df)} rows, columns: {list(input_df.columns)}')
else:
    print(f'WARNING: {CSV_INPUT} not found — will build from PDB filenames only')
    input_df = pd.DataFrame({'name': [os.path.splitext(os.path.basename(p))[0]
                                       for p in pdb_files]})

## 3. Contact analysis

In [ ]:
import numpy as np
from Bio import PDB
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

parser = PDB.PDBParser(QUIET=True)

def get_ca_atoms(chain):
    """Return {res_num: (coord, bfactor)} for all Cα atoms in chain."""
    atoms = {}
    for res in chain.get_residues():
        if res.get_id()[0] != ' ':
            continue
        res_num = res.get_id()[1]
        if 'CA' in res:
            ca = res['CA']
            atoms[res_num] = (ca.get_vector().get_array(), ca.get_bfactor())
    return atoms

def extract_sequence_from_chain(chain):
    """Extract 1-letter sequence from a Bio.PDB chain."""
    aa3to1 = {
        'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
        'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
        'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    }
    seen, seq = set(), []
    for res in chain.get_residues():
        if res.get_id()[0] != ' ':
            continue
        rid = res.get_id()[1]
        if rid not in seen:
            seen.add(rid)
            seq.append(aa3to1.get(res.get_resname(), 'X'))
    return ''.join(seq)

def analyze_complex(path):
    struct = parser.get_structure('s', path)
    model  = struct[0]
    chain_ids = [c.id for c in model]

    if TARGET_CHAIN not in chain_ids:
        return None, f'Target chain {TARGET_CHAIN} not found (chains: {chain_ids})'
    if BINDER_CHAIN not in chain_ids:
        return None, f'Binder chain {BINDER_CHAIN} not found (chains: {chain_ids})'

    target_ca = get_ca_atoms(model[TARGET_CHAIN])
    binder_ca = get_ca_atoms(model[BINDER_CHAIN])

    if not target_ca or not binder_ca:
        return None, 'Empty chain'

    epitope_contacts = defaultdict(set)
    contacting_binder_res = set()

    for b_res, (b_coord, _) in binder_ca.items():
        for t_res, (t_coord, _) in target_ca.items():
            dist = np.linalg.norm(b_coord - t_coord)
            if dist <= CONTACT_CUTOFF_A:
                contacting_binder_res.add(b_res)
                label = epitope_res_map.get(t_res)
                if label:
                    epitope_contacts[label].add(t_res)

    interface_plddt = (
        np.mean([binder_ca[r][1] for r in contacting_binder_res if r in binder_ca])
        if contacting_binder_res else 0.0
    )

    total_epitope_contacts = sum(len(v) for v in epitope_contacts.values())
    breakdown = ' | '.join(f'{lbl}: {len(res)}' for lbl, res in epitope_contacts.items())
    binder_seq = extract_sequence_from_chain(model[BINDER_CHAIN])

    return {
        'total_epitope_contacts': total_epitope_contacts,
        'epitope_breakdown':      breakdown,
        'total_contacts':         len(contacting_binder_res),
        'interface_plddt':        round(float(interface_plddt), 1),
        'binder_length':          len(binder_ca),
        'sequence_from_pdb':      binder_seq,
    }, None


if HAS_COMPLEX_PDB:
    print(f'Analyzing {len(pdb_files)} complex PDBs...')
    contact_records = {}
    for path in pdb_files:
        stem   = os.path.splitext(os.path.basename(path))[0]
        result, err = analyze_complex(path)
        if err:
            contact_records[stem] = {'error': err, 'total_epitope_contacts': 0,
                                     'interface_plddt': 0.0, 'epitope_breakdown': ''}
        else:
            contact_records[stem] = result
    print('Done.')
else:
    print('HAS_COMPLEX_PDB=False — skipping contact analysis (Forge binder-only mode)')
    print('Filter will use pLDDT threshold only.')
    contact_records = {}

## 4. Apply filter + merge with input CSV

In [ ]:
import shutil

results = []

for _, row in input_df.iterrows():
    name = str(row.get('name', ''))
    plddt = float(row.get('plddt', 0.0))
    iptm  = float(row.get('iptm',  0.5))
    seq   = str(row.get('sequence', ''))

    rec = {'name': name, 'sequence': seq, 'iptm': iptm, 'plddt': plddt,
           'method': METHOD, 'target_conformer': row.get('target_conformer', '')}

    if HAS_COMPLEX_PDB:
        # Find contact record by stem matching
        cr = contact_records.get(name) or contact_records.get(name + '_complex', {})
        if not cr:
            # Try partial match
            matches = [k for k in contact_records if name in k or k in name]
            cr = contact_records[matches[0]] if matches else {}

        epi_contacts   = cr.get('total_epitope_contacts', 0)
        iface_plddt    = cr.get('interface_plddt', 0.0)
        breakdown      = cr.get('epitope_breakdown', '')
        err            = cr.get('error', '')

        # Use sequence from PDB if not in input CSV
        if not seq and cr.get('sequence_from_pdb'):
            rec['sequence'] = cr['sequence_from_pdb']

        rec.update({'epitope_contacts': epi_contacts,
                    'epitope_breakdown': breakdown,
                    'interface_plddt_contact': iface_plddt})

        if err:
            verdict, reason = 'ERROR', err
        elif epi_contacts < MIN_EPITOPE_CONTACTS:
            verdict = 'DISCARD'
            reason  = f'only {epi_contacts} epitope contact(s) (need {MIN_EPITOPE_CONTACTS})'
        elif iface_plddt < MIN_INTERFACE_PLDDT:
            verdict = 'DISCARD'
            reason  = f'interface pLDDT={iface_plddt:.1f} < {MIN_INTERFACE_PLDDT}'
        else:
            verdict, reason = 'PASS', ''
    else:
        # Forge: filter on pLDDT only
        rec.update({'epitope_contacts': -1, 'epitope_breakdown': 'N/A (no complex PDB)',
                    'interface_plddt_contact': plddt})
        if plddt < MIN_PLDDT_FORGE:
            verdict = 'DISCARD'
            reason  = f'pLDDT={plddt:.1f} < {MIN_PLDDT_FORGE}'
        else:
            verdict, reason = 'PASS', ''

    rec['verdict'] = verdict
    rec['reason']  = reason
    results.append(rec)

df = pd.DataFrame(results)

n_pass    = (df['verdict'] == 'PASS').sum()
n_discard = (df['verdict'] == 'DISCARD').sum()
n_error   = (df['verdict'] == 'ERROR').sum()

print('=' * 50)
print(f'CONTACT FILTER — {METHOD.upper()}')
print('=' * 50)
print(f'  Total    : {len(df)}')
print(f'  PASS     : {n_pass}  ({100*n_pass/len(df):.0f}%)')
print(f'  DISCARD  : {n_discard}')
print(f'  ERROR    : {n_error}')
print()
print(df[df['verdict']=='PASS'][['name','iptm','plddt','epitope_contacts',
                                   'epitope_breakdown']].head(10).to_string(index=False))

## 5. QC plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
sizes  = [n_pass, n_discard] + ([n_error] if n_error else [])
labels = [f'PASS ({n_pass})', f'DISCARD ({n_discard})'] + \
         ([f'ERROR ({n_error})'] if n_error else [])
colors = ['#4CAF50', '#F44336', '#9E9E9E']
ax.pie(sizes, labels=labels, colors=colors[:len(sizes)], autopct='%1.0f%%', startangle=90)
ax.set_title(f'Filter result — {METHOD}')

ax = axes[1]
ax.hist(df[df['verdict']=='DISCARD']['plddt'], bins=20,
        color='#F44336', alpha=0.7, label='DISCARD')
ax.hist(df[df['verdict']=='PASS']['plddt'], bins=20,
        color='#4CAF50', alpha=0.7, label='PASS')
ax.axvline(MIN_PLDDT_FORGE if not HAS_COMPLEX_PDB else MIN_INTERFACE_PLDDT,
           color='black', ls='--', lw=1.5, label='threshold')
ax.set_xlabel('pLDDT'); ax.set_ylabel('Count')
ax.set_title('pLDDT distribution'); ax.legend(fontsize=8)

ax = axes[2]
if HAS_COMPLEX_PDB:
    pass_df = df[df['verdict']=='PASS']
    ax.scatter(pass_df['epitope_contacts'], pass_df['interface_plddt_contact'],
               c=pass_df['iptm'], cmap='RdYlGn', s=50, vmin=0.5, vmax=1.0)
    ax.set_xlabel('Epitope contacts'); ax.set_ylabel('Interface pLDDT')
    ax.set_title('Passing: epitope contacts vs interface pLDDT\n(color=ipTM)')
else:
    ax.scatter(df[df['verdict']=='PASS']['plddt'],
               df[df['verdict']=='PASS']['iptm'],
               color='#4CAF50', s=40, alpha=0.7)
    ax.set_xlabel('pLDDT'); ax.set_ylabel('ipTM')
    ax.set_title('Passing: pLDDT vs ipTM')

plt.suptitle(f'Binder contact filter — {METHOD}', fontsize=13)
plt.tight_layout()
plt.savefig(f'binder_filter_qc_{METHOD}.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save + download

In [ ]:
import zipfile
from google.colab import files as _colab_files

# Output filenames match what NB03 expects
passing_csv = f'{METHOD}_binder_filter_passing.csv'
all_csv     = f'{METHOD}_binder_filter_all.csv'

pass_df = df[df['verdict'] == 'PASS'].copy()

# Column order NB03 expects: name, sequence, iptm, plddt, method, ...
out_cols = ['name', 'sequence', 'iptm', 'plddt', 'method', 'target_conformer',
            'epitope_contacts', 'epitope_breakdown', 'interface_plddt_contact', 'verdict']
out_cols = [c for c in out_cols if c in pass_df.columns]

pass_df[out_cols].to_csv(passing_csv, index=False)
df[out_cols + ['reason']].to_csv(all_csv, index=False)

print(f'Passing CSV : {passing_csv}  ({len(pass_df)} rows)')
print(f'All CSV     : {all_csv}  ({len(df)} rows)')

# Copy passing PDB structures
pdb_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in pdb_files}
copied = 0
for name in pass_df['name']:
    src = pdb_lookup.get(name)
    if src:
        shutil.copy2(src, f'binders_passing/{name}.pdb')
        copied += 1
print(f'PDB files   : {copied} copied to binders_passing/')

# Zip for download
zip_name = f'{METHOD}_filter_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(passing_csv)
    zf.write(all_csv)
    zf.write(f'binder_filter_qc_{METHOD}.png')
    for fn in os.listdir('binders_passing'):
        zf.write(f'binders_passing/{fn}', f'binders_passing/{fn}')

print(f'\nDownloading {zip_name}')
print('\nNext:')
print(f'  1. Change METHOD to the other design tool and re-run with its zip')
print(f'  2. When both {METHOD}_binder_filter_passing.csv files are ready,')
print(f'     upload both to NB03 (binder comparison)')
_colab_files.download(zip_name)